[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/07_peptide_cyclic/07_peptide_lab.ipynb)


# 07 — 펩타이드·고리형(cyclotide) 실습

> 본문 [`07_peptide_cyclic.md`](07_peptide_cyclic.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 **여러분이 직접 돌린 결과**(`my_run/`)에서 계산합니다.
> **① 직접 설계 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 갑니다. 설계 셀은 NVIDIA GPU 전용이에요(CPU 폴백 없음) — Colab 이면 **런타임 → 런타임 유형 변경 → T4 GPU**.

## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `07_peptide_cyclic/` 로 이동합니다. 로컬에서 `07_peptide_cyclic/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "07_peptide_cyclic"
PIP_PKGS = "pandas matplotlib gemmi py3Dmol"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) 직접 돌려보기 — 내 결과 만들기

- 학습용 규모 `num_designs=8 --budget=4` (레퍼런스 결과는 num_designs=100)
- 소요 시간 실측 **585초**(최종 4개) — **24GB GPU · 가중치 캐시** 기준이에요. Colab T4 는 가속 커널이 꺼져 더 걸리고(T4 실측치 없음), 첫 실행은 가중치 ~6GB 다운로드가 더 붙어요.
- 건너뛰면 아래 분석이 커밋된 레퍼런스 결과로 이어집니다.

In [ ]:
SPEC, PROTOCOL = "example/cyclotide/3ivq.yaml", "peptide-anything"
NUM_DESIGNS, BUDGET = 8, 4

import shutil
OUT = MY.resolve()

def _gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return shutil.which("nvidia-smi") is not None

if not _gpu():
    print("GPU 런타임이 아니라 설계 실행을 건너뜁니다 — 아래 분석은 레퍼런스 결과로 이어집니다.")
    print("  · 직접 돌리려면 Colab [런타임 → 런타임 유형 변경 → T4 GPU] 후 이 셀을 다시 실행하세요.")
else:
    SRC = ADV_ROOT / ".boltzgen_src"          # 예제 명세·타깃 CIF 가 들어 있는 BoltzGen 소스
    if not SRC.exists():
        _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')
    if not _have("boltzgen"):
        _run(f'"{sys.executable}" -m pip -q install -e "{SRC}"')
    try:
        _run(f'cd "{SRC}" && boltzgen run {SPEC} --output "{OUT}" --protocol {PROTOCOL} '
             f'--num_designs {NUM_DESIGNS} --budget {BUDGET}')
        print("\n내 결과 →", OUT / "final_ranked_designs")
    except Exception as e:
        print("\n설계 실행이 도중에 멈췄어요 —", type(e).__name__)
        print("  · 이 셀을 다시 실행하면 같은 --output 산출물을 재사용해 이어서 끝냅니다(실측 763초 → 재개 486초).")
        print("  · GPU 메모리가 부족했다면 NUM_DESIGNS 4, BUDGET 2 로 줄이세요.")

## 2) 방금 그 명령이 요구한 것 — 입력 명세 (본문 7.3·7.4·7.5)

`example/cyclotide/3ivq.yaml` 은 짧지만 제약이 세 겹으로 걸려 있어요. 뒤에서 결과를 검증할 때 기준이 되니
먼저 읽고 갑니다.

```yaml
entities:
  - protein:
      id: B
      sequence: 3C8C6C5C3C1C2   # 3개+C+8개+C+6개+C+5개+C+3개+C+1개+C+2개 = 34잔기, Cys 6개
      cyclic: true              # 머리-꼬리 펩타이드 결합 자동 생성
  - file:
      path: 3ivq.cif
      include: [ { chain: { id: A } } ]
      structure_groups: "all"
constraints:
  - bond: { atom1: [B, 4, SG],  atom2: [B, 26, SG] }   # [체인, 잔기번호, 원자이름]
  - bond: { atom1: [B, 13, SG], atom2: [B, 30, SG] }
  - bond: { atom1: [B, 20, SG], atom2: [B, 32, SG] }
```

- `cyclic: true` — N말단과 C말단 사이에 펩타이드 결합이 자동으로 생겨요. 끝이 없으니 분해효소가 물 자리가 없죠.
- `constraints.bond` — 시스테인 6개를 3쌍으로 묶어 이황화(-S-S-)를 만듭니다. 이상 거리는 약 2.0~2.1Å 이에요.
- 서열 패턴 `3C8C6C5C3C1C2` — Cys 자리를 4·13·20·26·30·32 로 못 박고 나머지 28자리만 AI 가 채웁니다.

이 셋이 겹쳐야 cystine knot 이 완성돼요. 그러니 검증할 것도 셋이에요 — 서열, 고리, 이황화.

## 3) 이 실행이 실제로 밟은 스텝 (본문 7.2)

`peptide-anything` 은 inverse folding 단계에서 자유 시스테인을 억제해요 — 자유 Cys 가 엉뚱한 이황화나
응집을 만들기 때문이에요. 우리가 `constraints` 로 지정한 Cys 는 그와 별개로 배치되고요.
실제로 어떤 스텝이 돌았는지는 출력 폴더의 `steps.yaml` 에 그대로 적혀 있습니다.

In [ ]:
import pathlib
steps = pathlib.Path(find_one("steps.yaml", "data/cyclotide"))
print(steps.read_text().strip())

## 4) 최종 10개 메트릭 (본문 7.8)

스텝을 다 밟고 나온 결과예요. `num_tokens` 가 복합체 전체 크기(설계 펩타이드 + 타깃),
`num_design` 이 AI 가 실제로 채운 자리 수입니다.

In [ ]:
from boltzgen_viz import load_metrics, metrics_overview
import pandas as pd
CSV = str(find_one("final_designs_metrics_*.csv", "data/cyclotide"))
df = pd.read_csv(CSV)
print("최종 디자인", len(df), "개")
df[cols_in(df, "id", "final_rank", "design_ptm", "design_to_target_iptm",
           "min_design_to_target_pae", "filter_rmsd", "plip_hbonds_refolded",
           "num_design", "num_tokens")].sort_values("final_rank")

## 5) 메트릭 개요 그래프 (본문 7.8)

길이가 34aa 로 고정된 설계라 네 번째 패널은 길이 산점도 대신 H-bond 막대를 씁니다.

In [ ]:
rows = load_metrics(CSV)
FIG = my_fig("07_cyclotide_metrics.png")
metrics_overview(rows, "Cyclotide (3ivq) — Design Metrics Overview", FIG, panel4="hbonds")
from IPython.display import Image; Image(FIG)

## 6) 검증 ① 서열 — 제약 자리가 살아남았나 (본문 7.5)

그래프는 "얼마나 좋은가"를 보여줄 뿐, "요구한 대로 만들어졌는가"는 답하지 않아요. 먼저 서열부터 봅니다.
길이가 34aa 인지, 그리고 명세가 Cys 로 못 박은 4·13·20·26·30·32 자리가 그대로인지 세어 보죠.
(`num_design`=28 은 34자리에서 Cys 6자리를 뺀, AI 가 채운 자리 수예요.)

In [ ]:
from collections import Counter
SEQ_LEN   = 34                                 # 3+1+8+1+6+1+5+1+3+1+1+1+2
CYS_SITES = [4, 13, 20, 26, 30, 32]            # 명세가 Cys 로 고정한 자리
d = df.sort_values("final_rank")
n_len = n_sites = 0
for _, r in d.iterrows():
    s = str(r["designed_chain_sequence"])
    pos = [i + 1 for i, a in enumerate(s) if a == "C"]
    kept = all(p in pos for p in CYS_SITES)
    n_len += int(len(s) == SEQ_LEN)
    n_sites += int(kept)
    mark = "" if kept else "   <-- 제약 자리 누락"
    print(f"rank{int(r['final_rank']):>2} {str(r['id']):9s} len={len(s):3d} Cys={len(pos):2d} @ {pos}{mark}")
print(f"\n길이 {SEQ_LEN}aa {n_len}/{len(d)} | 제약 Cys 6자리 전부 보존 {n_sites}/{len(d)}")
print("Cys 총개수 분포", dict(sorted(Counter(str(x).count("C")
                                        for x in d["designed_chain_sequence"]).items())))

합격선은 길이와 제약 자리예요. 6자리가 전부 살아 있으면 이황화 3쌍을 만들 재료가 갖춰진 거고,
그 밖의 자리에 Cys 가 더 붙은 건 생성 다양성이라 그 자체로는 탈락 사유가 아니에요.
다만 짝 없는 Cys 는 응집 위험이니, 실제로 어느 쌍이 결합했는지는 8절에서 거리로 확인합니다.

## 7) 검증 ② 구조 — 고리가 닫혔나 (본문 7.3)

`cyclic: true` 가 지켜졌다면 N말단의 N 과 C말단의 C 가 펩타이드 결합 거리(약 1.3Å)만큼 붙어 있어야 해요.
최종 CIF 를 gemmi 로 열어 직접 잽니다. 출력 CIF 의 사슬 라벨은 명세와 다를 수 있어서,
라벨 대신 설계 서열과 일치하는 사슬을 찾습니다.

In [ ]:
import gemmi
top = df.sort_values("final_rank").iloc[0]
cif = find_one(f"final*designs/*{top['id']}*.cif", "data/cyclotide")
model = gemmi.read_structure(str(cif))[0]

want = str(top["designed_chain_sequence"])
def chain_seq(ch):
    return gemmi.one_letter_code([r.name for r in ch]).upper()
hit = [ch for ch in model if chain_seq(ch) == want]
design = hit[0] if hit else min(model, key=lambda ch: len(ch))   # 폴백은 가장 짧은 사슬
res = list(design)

print(cif.name, "| 사슬 구성", {ch.name: len(ch) for ch in model})
print(f"설계 사슬 = {design.name} ({len(res)}잔기), 나머지가 타깃")

n_at, c_at = res[0].find_atom("N", "*"), res[-1].find_atom("C", "*")
if n_at is None or c_at is None:
    print("\n말단 백본 원자가 없어 고리화 판정을 건너뜁니다.")
else:
    d_nc = n_at.pos.dist(c_at.pos)
    verdict = "펩타이드 결합 = 고리 닫힘" if d_nc < 1.6 else "떨어져 있음 = 선형"
    print(f"\nN말단 N ↔ C말단 C 거리 {d_nc:.2f} A → {verdict}")

## 8) 검증 ③ 구조 — 이황화 3쌍이 맺혔나 (본문 7.4)

고리가 닫혔으면 이제 매듭이에요. 같은 사슬 안 모든 Cys 쌍의 SG–SG 거리를 재서
이상값 2.0~2.1Å 근처인 쌍을 찾습니다. 서열상 두 Cys 사이 간격도 함께 봐요 — 권장은 3~15잔기인데,
고리형이라 반대 방향으로 도는 경로가 더 짧으면 그쪽을 씁니다.

In [ ]:
import itertools
sg = [(r.seqid.num, r.sole_atom("SG").pos) for r in res
      if r.name == "CYS" and r.find_atom("SG", "*") is not None]
L = len(res)
pairs = sorted(((a[0], b[0], a[1].dist(b[1])) for a, b in itertools.combinations(sg, 2)),
               key=lambda x: x[2])
bonded = [p for p in pairs if p[2] < 2.5]
print(f"SG 원자 {len(sg)}개 | 이상 SG-SG 거리 2.0~2.1 A")
for i, j, dist in bonded + [p for p in pairs if p[2] >= 2.5][:2]:
    step = abs(j - i)
    gap  = min(step - 1, L - step - 1)          # 고리 기준 최단 간격(사이에 낀 잔기 수)
    if dist < 2.5:
        note = "권장 3~15 안" if 3 <= gap <= 15 else "권장 범위 밖"
        print(f"  CYS{i:>3}-CYS{j:<3} {dist:5.2f} A  이황화 | 사이 잔기 {gap:2d} ({note})")
    else:
        print(f"  CYS{i:>3}-CYS{j:<3} {dist:5.2f} A  결합 아님")
print(f"\n이황화 {len(bonded)}쌍 형성 — 명세가 요구한 건 Cys4-26, 13-30, 20-32 세 쌍이에요.")

판정은 거리로 합니다. 2.0~2.1Å 근처면 결합, 8Å처럼 크게 벌어졌으면 안 닫힌 거예요.
그때 손볼 곳은 구조가 아니라 서열 패턴이에요 — 두 Cys 를 서열상 더 가깝게(`5C20C5` → `5C10C5`) 좁히면
닫힐 확률이 올라갑니다. 간격 3~15잔기는 그 설계 단계의 권장 범위이지, 이미 닫힌 결합의 합격선은 아니에요.

제약 3쌍이 모두 2.5Å 미만으로 나왔다면 cystine knot 이 완성된 겁니다.

## 9) 고리와 매듭을 눈으로 — rank 1 디자인 3D (본문 7.3·7.4)

7)·8)절은 거리를 **숫자로** 쟀어요. 같은 것을 그림으로 확인합니다.

**주황이 설계 cyclotide, 파랑이 타깃**이고, Cys 의 **SG 원자를 노란 공**으로 띄웠어요.
두 개씩 붙어 있는 노란 공 쌍이 곧 이황화 결합입니다. N말단·C말단은 굵은 stick 으로 표시했고요.
마우스로 회전·확대할 수 있어요(휠 = 줌, 드래그 = 회전). 구조가 안 보이면 `py3Dmol` 설치 로그를 확인하고 셀을 한 번 더 실행하세요.

In [ ]:
import importlib.util, sys, pathlib
for _pkg in ("py3Dmol", "gemmi"):                       # 없으면 그 자리에서 설치
    if importlib.util.find_spec(_pkg) is None:
        _run(f'"{sys.executable}" -m pip -q install {_pkg}')
import py3Dmol, gemmi

C_DESIGN  = ["#e8883a", "#c0392b"]              # 설계 사슬 — 주황·빨강
C_TARGET  = ["#3477b5", "#7f8c8d", "#8e44ad"]   # 타깃 단백질 — 파랑·회색·보라
C_NUCLEIC = "#1aa090"                           # 타깃 핵산 — 청록

def chain_seq3d(ch):
    """사슬의 한 글자 서열(폴리머 잔기만)."""
    poly = ch.get_polymer()
    return gemmi.one_letter_code([r.name for r in poly]).upper() if len(poly) else ""

def chain_kind3d(ch):
    """protein / dna / rna / hetero(리간드·금속) 판별."""
    poly = ch.get_polymer()
    if not len(poly):
        return "hetero"
    t = str(poly.check_polymer_type())
    return "dna" if "Dna" in t else "rna" if "Rna" in t else "protein"

def load_design(pattern, ref, row=None,
                seq_cols=("full_sequence_1", "full_sequence_2", "designed_chain_sequence")):
    """최종 CIF 를 찾아 열고 (pdb문자열, model, 설계사슬, 타깃사슬, 사슬→CSV컬럼) 을 돌려줍니다.
    설계 사슬은 CSV 의 설계 서열과 사슬 서열이 같은 것으로 정해요 — 사슬 라벨을 하드코딩하지 않습니다."""
    cif = find_one(pattern, ref)                        # [내 결과]/[레퍼런스] 를 스스로 출력합니다
    print("띄울 구조 :", cif, "→", "내 결과 my_run/" if "my_run" in str(cif) else "커밋된 레퍼런스 data/")
    st = gemmi.read_structure(str(cif)); st.setup_entities()
    model = st[0]

    want = {}
    if row is not None:
        for col in seq_cols:
            v = row[col] if col in getattr(row, "index", []) else None
            if isinstance(v, str) and v.strip():
                want.setdefault(v.strip().upper(), col)   # 같은 서열이면 먼저 온 컬럼을 씁니다
    match  = {ch.name: want[chain_seq3d(ch)] for ch in model if chain_seq3d(ch) in want}
    design = list(match)
    if not design:                                        # 폴백 — 설계물은 보통 가장 짧은 단백질 사슬
        prot = [ch for ch in model if chain_kind3d(ch) == "protein"]
        design = [min(prot, key=lambda c: len(c.get_polymer())).name] if prot else []
        print("  (CSV 서열과 일치하는 사슬이 없어 가장 짧은 단백질 사슬을 설계물로 봅니다)")
    target = [ch.name for ch in model if ch.name not in design]

    for ch in model:
        n = len(ch.get_polymer()) or len(ch)
        print(f"  사슬 {ch.name:<2s} {'설계' if ch.name in design else '타깃'} "
              f"| {chain_kind3d(ch):<7s} {n:>4d} | {chain_seq3d(ch)[:28]}")
    return st.make_pdb_string(), model, design, target, match

def complex_view(pdb, model, design, target, width=760, height=540):
    """설계 사슬(주황)·타깃 단백질(파랑)·핵산(청록) cartoon + 비폴리머(리간드 stick·이온 sphere)."""
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb, "pdb")
    view.setStyle({}, {"cartoon": {"color": "#cfd6dd"}})
    ti = 0
    for ch in model:
        kind = chain_kind3d(ch)
        if kind == "hetero":
            continue
        if ch.name in design:
            color = C_DESIGN[design.index(ch.name) % len(C_DESIGN)]
        elif kind in ("dna", "rna"):
            color = C_NUCLEIC
        else:
            color = C_TARGET[ti % len(C_TARGET)]; ti += 1
        style = {"cartoon": {"color": color}}
        if kind in ("dna", "rna"):
            style["stick"] = {"color": color, "radius": 0.12}   # 염기까지 보이게
        view.setStyle({"chain": ch.name}, style)
    for name, natoms in {r.name: len(r) for ch in model for r in ch if r.het_flag == "H"}.items():
        if natoms == 1:                                   # 금속·이온
            view.addStyle({"resn": name}, {"sphere": {"color": "#6c5ce7", "radius": 0.9}})
        else:                                             # 리간드
            view.addStyle({"resn": name}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.22}})
    view.setBackgroundColor("white")
    return view

def designed_segments(full, des, min_len=3):
    """이어붙여 저장된 재설계 구간(des)을 전체 서열(full) 위에 되짚어 (시작,끝) 1-based 목록으로.
    CDR 번호를 노트북에 적어 두지 않고 결과 CSV 에서 되찾기 위한 것입니다."""
    full, des = str(full).upper(), str(des).upper()
    out, i, pos = [], 0, 0
    while i < len(des):
        best, at = 0, -1
        for L in range(len(des) - i, min_len - 1, -1):
            j = full.find(des[i:i + L], pos)
            if j >= 0:
                best, at = L, j
                break
        if best == 0:
            i += 1
            continue
        out.append((at + 1, at + best))
        i += best; pos = at + best
    return out

def seg_resi(model, chain_id, segs):
    """서열 위 구간 → 그 사슬의 실제 잔기 번호 목록(3Dmol 의 resi 선택용)."""
    ch = {c.name: c for c in model}[chain_id]
    nums = [r.seqid.num for r in ch.get_polymer()]
    return [n for a, b in segs for n in nums[a - 1:b]]

# 사슬을 찾아준 컬럼 → 그 사슬의 '재설계 구간' 컬럼
DES_COL = {"full_sequence_1": "designed_sequence_1", "full_sequence_2": "designed_sequence_2",
           "designed_chain_sequence": "designed_sequence"}

top = df.sort_values("final_rank").iloc[0]
print("rank 1 디자인 :", top["id"])
pdb, model, DESIGN, TARGET, MATCH = load_design(
    f"final*designs/*{top['id']}*.cif", "data/cyclotide", row=top)

view = complex_view(pdb, model, DESIGN, TARGET)
chains = {ch.name: ch for ch in model}
for cid in DESIGN:
    res = list(chains[cid])
    cys = [r.seqid.num for r in res if r.name == "CYS"]
    view.addStyle({"chain": cid, "resi": cys},
                  {"stick": {"colorscheme": "yellowCarbon", "radius": 0.16}})
    view.addStyle({"chain": cid, "atom": "SG"},           # 이황화가 보이도록 SG 만 공으로
                  {"sphere": {"color": "#f1c40f", "radius": 0.6}})
    # 머리-꼬리 고리화 자리 — 첫 잔기와 마지막 잔기(하드코딩 없이 사슬 양 끝)
    view.addStyle({"chain": cid, "resi": [res[0].seqid.num]},
                  {"stick": {"color": "#2980b9", "radius": 0.3}})
    view.addStyle({"chain": cid, "resi": [res[-1].seqid.num]},
                  {"stick": {"color": "#c0392b", "radius": 0.3}})
    print(f"  설계 사슬 {cid} — Cys {len(cys)}개 @ {cys} | N말단 {res[0].seqid.num}(파랑) "
          f"· C말단 {res[-1].seqid.num}(빨강)")

view.zoomTo({"chain": DESIGN})        # 34잔기 펩타이드가 작아서 설계 사슬 기준으로 당깁니다
view.show()

print("\n무엇을 봐야 하나")
print("  · 노란 공(SG) 6개가 3쌍으로 붙어 있으면 cystine knot 이 맺힌 거예요 — 8)절 거리표와 같은 이야기입니다.")
print("  · 파랑(N말단)과 빨강(C말단) stick 이 맞닿아 있으면 고리가 닫힌 것(7)절 실측 1.31 A).")
print("  · 주황 펩타이드가 파랑 타깃 표면 홈에 얹혀 있으면 결합 포즈도 정상이에요.")
try:
    print("\n8)절이 잰 이황화 —", ", ".join(f"CYS{i}-CYS{j} {d:.2f}A" for i, j, d in bonded) or "없음")
except NameError:
    print("\n(8)절 셀을 먼저 실행하면 여기서 이황화 거리도 같이 출력됩니다)")

## 10) 더 좁히고 싶을 때 — 2차구조와 결합부위 (본문 7.6·7.7)

여기까지가 기본 제약이고, 결과가 마음에 안 들면 두 가지를 더 걸 수 있어요.

2차구조를 강제하려면 `secondary_structure` 를 씁니다. 평평한 결합면에는 sheet, 깊은 groove 에는 helix 가
유리해요. sheet 는 최소 3~4잔기 이상이어야 형성되고, 너무 짧게 지정하면 loop 로 끝나요.

```yaml
- protein:
    id: B
    sequence: 1C11..16C1
    secondary_structure:
        sheet: 1,3..11          # 이 잔기들을 beta-sheet 로
```

붙을 자리를 지정하려면 타깃 쪽에 `binding_types` 를 겁니다.

```yaml
- file:
    path: target.cif
    include: [ { chain: { id: A } } ]
    binding_types:
      - chain: { id: A, binding: 343,344,251 }
    structure_groups: "all"
```

둘 다 선택 기능이라 굳이 강제하지 않아도 AI 가 알아서 최적화해요 — 특정 구조나 자리가 꼭 필요할 때만 쓰세요.
`binding_types` 를 걸었다면 결과 CSV 의 인터페이스 접촉 잔기를 확인해 지정한 자리에 실제로 붙었는지 검증하고요.

## 11) 종합 판정과 실험 후보 (본문 7.8·7.9)

제약이 다 지켜졌다면 이제 순위를 읽을 차례예요. 읽는 원칙은 셋입니다.

- ipTM·pTM·PAE 가 함께 순위를 좌우해요. 어느 하나만 보고 고르지 않습니다.
- RMSD 는 자기일관성 — 낮을수록 "설계한 모양이 서열로 재현"된 거예요.
- H-bond 수는 순위를 결정하지 않아요. 레퍼런스에서는 H-bond 가 10개로 가장 많은 rank 3 이 최종 3위였어요.

실험 후보는 하나만 고르지 말고 성격이 다른 셋을 함께 보내는 게 정석이에요(본문 7.9).

In [ ]:
d = df.sort_values("final_rank")
pick = [("결합력(ipTM 최고)",     d["design_to_target_iptm"].idxmax()),
        ("구조 안정성(pTM 최고)", d["design_ptm"].idxmax()),
        ("자기일관성(RMSD 최저)", d["filter_rmsd"].idxmin())]
for label, idx in pick:
    r = d.loc[idx]
    print(f"rank{int(r['final_rank']):>2} {str(r['id']):9s} "
          f"ipTM {r['design_to_target_iptm']:.3f} | pTM {r['design_ptm']:.3f} "
          f"| RMSD {r['filter_rmsd']:.2f} A   <- {label}")
hb = d.loc[d["plip_hbonds_refolded"].idxmax()]
print(f"\nRMSD 최댓값 {d['filter_rmsd'].max():.2f} A"
      f" | H-bond 최다는 rank {int(hb['final_rank'])} ({int(hb['plip_hbonds_refolded'])}개)")

34잔기짜리 작은 펩타이드라 SPPS(고상 펩타이드 합성)로 만들 수 있어요. 실험 경로는 이렇습니다.

1. SPPS 로 선형 펩타이드 합성 (약 2주)
2. Native chemical ligation 으로 고리화
3. Oxidative folding 으로 이황화 3쌍 형성
4. Mass spec + NMR 로 구조 검증
5. SPR 로 결합 친화도 측정

이 챕터에서 확인한 건 "메트릭보다 제약 준수가 먼저"라는 순서예요. 서열(6절)·고리(7절)·이황화(8절)가
모두 통과한 뒤에야 ipTM 순위가 의미를 가집니다. 다음 Ch.08 에서는 제약이 훨씬 많은 항체 Fab 을 다뤄요 —
거기서는 재설계 영역(CDR)만 바뀌고 framework 는 그대로여야 한다는 새 합격선이 붙습니다.